In the notebook ```sentence_encoder_test.ipynb``` we did some basic visualization of encoding the distributions of winners vs nonwinners. However, the plots that we got seem to indicate that there was not much substantial difference between the winners and nonwinners. 

In order to test if there is any actual difference, we will perform some hypothesis test.

In [3]:
import pandas as pd
import numpy as np

In [4]:
from sentence_transformers import SentenceTransformer

transformer = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
books = pd.read_csv('../../data/final_book_dataset_cleaned.csv', sep = '\t', 
dtype = {
    'isbn' : 'str', # do this explicitly to avoide getting a warning by the interpreter
    'author_birthyear' : 'Int64', # we have to explicitly do this to avoid pandas implicitly casting as float
    'title_id' : 'Int64'
})

books = books.dropna() # we do this in order to remove potential outliers that may skew our data incorrectly

In [6]:
sample = books[(2000 <= books.release_year) & (books.release_year <= 2013)]
sample

,title_id,title,author,release_year,release_date,first_publisher,author_birthyear,author_birthplace,isbn,book_synopsis,tags,hugo,locus
30045,741561,Bad Dreams,Anne Fine,2000,2000-06-00,Delacorte Press,1947,"Leicester, Leicestershire, England, UK",0385327579,"Despite her preference for books over friends,...",[],False,False
30049,918449,Mr. Mee,Andrew Crumey,2000,2000-05-00,Picador,1961,"Kirkintilloch, Dunbartonshire, Scotland, UK",0330376802,A New York Times Notable Book of the Year In t...,[],False,False
30053,739969,Whispers in the Sand,Barbara Erskine,2000,2000-09-00,HarperCollins (UK),1944,"Nottingham, Nottinghamshire, England, UK",000225784X,"Recently divorced, Anna decides to cheer herse...","['Fiction', 'Fantasy', 'Adventure', 'Romance',...",False,False
30056,729397,In the Country of the Young,Lisa Carey,2000,2000-11-00,William Morrow / HarperCollins,1970,"Boston, Massachusetts, USA",0380976757,"On a stormy November night in 1848, a ship car...","['Fantasy', 'Fiction', 'History']",False,False
30072,873545,Trouble on Tattooine,Dave Wolverton,2000,2000-04-00,Scholastic,1957,"Springfield, Oregon, USA",043910145X,Anakin Skywalker and his friends are in big tr...,"[""Children's stories"", 'Adventure']",False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
73334,1633699,Strikeforce,Nick James,2013,2013-10-08,Flux,1983,"Portland, Oregon, USA",9780738736372.0,As the alien Authority attacks Earth using a t...,['young-adult sf'],False,False
73341,1601405,Storm Force,Susannah Sandlin,2013,2013-03-19,Montlake Romance,1956,"Winfield, Alabama, USA",9781477857571.0,When a bomb explodes inside a Houston high-ris...,"['paranormal romance', 'paranormal suspense', ...",False,False
73342,1632034,Treecat Wars,"Jane Lindskold, David Weber",2013,2013-10-01,Baen Books,1962,"Washington, District of Columbia, USA",9781451639339.0,"The fires are out, but the trouble’s just begi...","['science fiction', 'young-adult sf', 'Science...",False,False
73345,1600519,Dragonwitch,Anne Elisabeth Stengl,2013,2013-07-15,Bethany House Publishers,1986,"McChord AFB, Washington, USA",9780764210273.0,A New Tale Is Added to this Christy Award-Winn...,"['religious fantasy', 'young-adult fantasy']",False,False


In [7]:
locus = sample[sample.locus]
nonlocus = sample[~(sample.locus)]

hugo = sample[sample.hugo]
nonhugo = sample[~(sample.hugo)]

In [8]:
locus_embed = transformer.encode(np.array(locus.book_synopsis))

In [9]:
locus_embed

array([[-0.10164671,  0.00948464, -0.0162692 , ..., -0.14832214,
         0.03050093,  0.06897666],
       [-0.06895616,  0.14146984,  0.01198744, ...,  0.0048253 ,
        -0.03022185,  0.04948013],
       [ 0.07926937, -0.0495549 ,  0.0295058 , ...,  0.02366009,
        -0.00610184,  0.0631576 ],
       ...,
       [-0.02824035,  0.02276585,  0.01558279, ..., -0.02006573,
         0.05978582,  0.0570555 ],
       [-0.09500596,  0.04716654, -0.04036315, ..., -0.03036787,
        -0.05574735, -0.03680762],
       [-0.07830554,  0.00356591, -0.05418869, ..., -0.05396573,
         0.03705451,  0.04993298]], shape=(407, 384), dtype=float32)

In [10]:
nonlocus_embed = transformer.encode(np.array(nonlocus.book_synopsis))

In [11]:
nonlocus_embed

array([[-0.02888517, -0.05045382,  0.09129327, ...,  0.07984006,
        -0.04837752, -0.01610985],
       [-0.09443668, -0.04422925,  0.00704637, ..., -0.02314086,
         0.00194411, -0.01518035],
       [-0.10260566,  0.0746371 ,  0.05064679, ...,  0.04494892,
        -0.0150765 , -0.04994808],
       ...,
       [ 0.03713535,  0.0261771 ,  0.05042421, ..., -0.07394161,
        -0.00733223,  0.00755602],
       [-0.0361672 ,  0.07856764,  0.00799091, ..., -0.12063682,
         0.02614656,  0.00224244],
       [-0.00961149,  0.04051654, -0.03969159, ..., -0.05296774,
        -0.06586707,  0.07554308]], shape=(8217, 384), dtype=float32)

As a first sanity check, we should check to see if the difference between the sample means of these two arrays are sufficiently apart.

In [12]:
locus_mean = locus_embed.mean(axis=0)
nonlocus_mean = nonlocus_embed.mean(axis=0)

In [ ]:
np.sqrt(
    np.square(locus_mean - nonlocus_mean).sum()
    )

np.float32(0.12571232)

This difference seems reasonably apart, so let's try to proceed to a hypothesis test.

We have two samples ```locus_embed``` and ```nonlocus_embed```. We want to see if these two samples come from the same multivariate distribution. It seems that one way to do this is to apply Hotelling's $T^2$ hypothesis test.

We assume somehow that the result of our embeddings are normal distributions, this test statistics should be valid enough to determine if these two distributions are actually different. We first apply this test naively, and then maybe later we can try some bootstrapping method.

In [ ]:
# %pip install scikit-fda # run once to install scikit-fda which implements the T^2 test

In [17]:
from skfda import FDataGrid
from skfda.inference.hotelling import hotelling_test_ind

In [18]:
fda_locus = FDataGrid(locus_embed)
fda_nonlocus = FDataGrid(nonlocus_embed)

In [19]:
t2_stat, p_val = hotelling_test_ind(fda_locus, fda_nonlocus, n_reps = 100)

print("T^2", t2_stat)
print('p value', p_val)

T^2 1051.8089445856922
p value 0.0
